# NII Rate/Volume Decomposition — Greek Systemic Banks

Splits the year-on-year change in net interest income (NII) into:
- **Rate (margin) effect**: change in NIM applied to last year's average assets, the part linked to ECB rate moves
- **Volume effect**: last year's NIM applied to the change in average assets
- **Mix effect**: the interaction of the two (small)

$$\Delta \text{NII} = \underbrace{\Delta \text{NIM} \times \bar{A}_{t-1}}_{\text{Rate}} + \underbrace{\text{NIM}_{t-1} \times \Delta \bar{A}}_{\text{Volume}} + \underbrace{\Delta \text{NIM} \times \Delta \bar{A}}_{\text{Mix}}$$

where $\text{NIM} = \text{NII} / \bar{A}$ and $\bar{A}$ is average total assets (the same NIM as in `kpis_final.csv`). Rate + Volume + Mix equals the actual ΔNII exactly.

**Rate context:** ECB deposit facility rate, yearly average (`data/manual/ecb_deposit_rate.csv`).
**Basis:** 2022 figures are the restated comparatives of the FY2023 reports, so 2022→2023 compares like with like (for example Alpha's Romanian business is treated as discontinued in both years).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(kpis)} bank-year rows.')

Loaded 12 bank-year rows.


In [2]:
# ── ECB deposit facility rate (yearly average) ───────────────────────────────
ecb = pd.read_csv(DATA_DIR.parent / 'manual' / 'ecb_deposit_rate.csv')
ecb_rate = dict(zip(ecb['year'], ecb['dfr_avg']))
source = ecb['source'].iloc[0]
print('ECB deposit facility rate, yearly average (%):', ecb_rate)

ECB deposit facility rate, yearly average (%): {2022: 0.08, 2023: 3.31, 2024: 3.73}


In [3]:
# ── NIM × average assets decomposition ───────────────────────────────────────
df = kpis[['bank','year','nii','avg_total_assets']].copy()
df['nim_exact'] = df['nii'] / df['avg_total_assets']   # unrounded, so the walk adds up exactly

transitions = [(2023, 2022), (2024, 2023)]
rows = []
for bank in BANKS:
    bdf = df[df['bank'] == bank].set_index('year')
    for yr, yp in transitions:
        nim_t, nim_p = bdf.loc[yr, 'nim_exact'], bdf.loc[yp, 'nim_exact']
        a_t, a_p     = bdf.loc[yr, 'avg_total_assets'], bdf.loc[yp, 'avg_total_assets']
        nii_t, nii_p = bdf.loc[yr, 'nii'], bdf.loc[yp, 'nii']
        d_nim, d_a = nim_t - nim_p, a_t - a_p
        rows.append({
            'bank': bank, 'year': yr,
            'nii_prev': nii_p, 'nii_t': nii_t, 'delta_nii': nii_t - nii_p,
            'rate_effect': d_nim * a_p,
            'volume_effect': nim_p * d_a,
            'mix_effect': d_nim * d_a,
            'd_nim_pp': d_nim * 100,
            'd_assets_pct': d_a / a_p * 100,
        })

decom = pd.DataFrame(rows)
decom['check_sum'] = decom[['rate_effect','volume_effect','mix_effect']].sum(axis=1)
decom['mismatch']  = (decom['check_sum'] - decom['delta_nii']).abs()
decom['rate_share'] = decom['rate_effect'] / decom['delta_nii'] * 100

print('Decomposition (€ million):')
print(decom[['bank','year','delta_nii','rate_effect','volume_effect','mix_effect','mismatch']].round(1).to_string(index=False))

Decomposition (€ million):
        bank  year  delta_nii  rate_effect  volume_effect  mix_effect  mismatch
    Eurobank  2023      694.0        668.0           17.9         8.1       0.0
    Eurobank  2024      333.0         60.4          265.3         7.4       0.0
  Alpha Bank  2023      485.2        495.5           -7.3        -3.1       0.0
  Alpha Bank  2024      -14.0         53.6          -65.5        -2.1       0.0
Piraeus Bank  2023      650.0        694.3          -29.3       -15.0       0.0
Piraeus Bank  2024       85.0         13.0           71.6         0.5       0.0
         NBG  2023      894.0       1032.9          -79.2       -59.7       0.0
         NBG  2024       93.0        142.7          -46.8        -2.9       0.0


In [4]:
# ── Chart 1: Waterfall Charts per Bank ────────────────────────────────────────
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    vertical_spacing=0.18, horizontal_spacing=0.10,
)

BAR_COLORS = {
    'rate_effect':   '#00CC96',
    'volume_effect': '#636EFA',
    'mix_effect':    '#AB63FA',
}
EFFECT_LABELS = {
    'rate_effect': 'Rate/Spread (ΔNIM × prior assets)',
    'volume_effect': 'Volume (prior NIM × ΔAssets)',
    'mix_effect': 'Mix (ΔNIM × ΔAssets)',
}
effects = ['rate_effect', 'volume_effect', 'mix_effect']

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    bdf = decom[decom['bank'] == bank].sort_values('year')
    x_labels = [f"{int(r['year'])-1}→{int(r['year'])}" for _, r in bdf.iterrows()]

    for effect in effects:
        y_vals = bdf[effect].values
        fig1.add_trace(
            go.Bar(
                x=x_labels, y=y_vals,
                name=EFFECT_LABELS[effect],
                marker_color=BAR_COLORS[effect],
                showlegend=(bi == 0), legendgroup=effect,
                text=[f'{v:+.0f}' for v in y_vals], textposition='outside',
            ),
            row=row + 1, col=col + 1,
        )

    fig1.add_trace(
        go.Scatter(
            x=x_labels, y=bdf['delta_nii'].values,
            mode='markers+text',
            marker=dict(symbol='diamond', size=12, color='white'),
            text=[f'Δ{v:+.0f}m' for v in bdf['delta_nii'].values],
            textposition='top center',
            name='Total ΔNII', showlegend=(bi == 0), legendgroup='delta',
        ),
        row=row + 1, col=col + 1,
    )

fig1.update_layout(
    title_text='<b>NII Rate vs Volume Waterfall</b> | ΔNIM × Assets decomposition (€ million)',
    title_font_size=16, template='plotly_dark', height=620, barmode='relative',
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center', font_size=11),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig1.show()

In [5]:
# ── Chart 3: ECB rate context ─────────────────────────────────────────────────
fig3 = make_subplots(rows=1, cols=2, subplot_titles=['ECB deposit rate (yearly average)', 'Sector NII vs ECB rate'])

fig3.add_trace(
    go.Bar(x=list(ecb_rate.keys()), y=list(ecb_rate.values()),
           marker_color=['#636EFA','#EF553B','#00CC96'],
           text=[f'{v:.2f}%' for v in ecb_rate.values()], textposition='outside'),
    row=1, col=1,
)

sector_nii = kpis.groupby('year')['nii'].sum().reset_index()
fig3.add_trace(
    go.Scatter(x=[ecb_rate[y] for y in sector_nii['year']], y=sector_nii['nii'].values,
               mode='markers+text',
               marker=dict(size=14, color=['#636EFA','#EF553B','#00CC96']),
               text=[str(y) for y in sector_nii['year']],
               textposition='top center'),
    row=1, col=2,
)

fig3.update_layout(
    title_text='<b>ECB Rate Context</b> | Sector NII against the ECB deposit rate',
    template='plotly_dark', height=350, showlegend=False,
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig3.update_xaxes(title_text='Year', row=1, col=1)
fig3.update_yaxes(title_text='ECB deposit rate (%)', row=1, col=1)
fig3.update_xaxes(title_text='ECB deposit rate (%)', row=1, col=2)
fig3.update_yaxes(title_text='Sector NII (€m)', row=1, col=2)
fig3.show()

In [6]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(decom) == 8, f'Expected 8 rows (4 banks × 2 transitions), got {len(decom)}'
max_mismatch = decom['mismatch'].max()
assert max_mismatch < 0.5, f'Decomposition mismatch: {max_mismatch:.4f}m'

r23 = decom[decom['year'] == 2023]
assert (r23['rate_effect'] > 0).all(), 'Expected a positive rate effect for every bank in 2023'

print('All checks passed.')
print(f'   Max decomposition mismatch: {max_mismatch:.6f} €m')
print(f'   2022→2023: sector ΔNII €{r23["delta_nii"].sum():,.0f}m, of which rate effect '
      f'€{r23["rate_effect"].sum():,.0f}m ({r23["rate_effect"].sum()/r23["delta_nii"].sum()*100:.0f}%)')
r24 = decom[decom['year'] == 2024]
print(f'   2023→2024: sector ΔNII €{r24["delta_nii"].sum():,.0f}m, rate effect €{r24["rate_effect"].sum():,.0f}m, '
      f'volume effect €{r24["volume_effect"].sum():,.0f}m')
print(f'   ECB rate source: {source}')

All checks passed.
   Max decomposition mismatch: 0.000000 €m
   2022→2023: sector ΔNII €2,723m, of which rate effect €2,891m (106%)
   2023→2024: sector ΔNII €497m, rate effect €270m, volume effect €225m
   ECB rate source: ECB key interest rates (ecb.europa.eu)
